# Receptive & projective field audit

Audit the two influence-geometry directions on a real ResNet execution.

**Surfaces covered:** `tl.receptive_field`, `Op.receptive_field`, `Op.projective_field`, `.at()`, `.gradient()`, `.check()`, `.show()`, `Trace.receptive_fields()`, `Trace.projective_fields()`, `cross_validate()`, `verify()`, `node_spec()`, `source=`, `direction=`, `target=`, and `tl.validate(..., scope="receptive_field")`.

In [ ]:
from pathlib import Path
import sys

repo = Path.cwd().resolve()
while not (repo / "pyproject.toml").exists():
    repo = repo.parent
sys.path.insert(0, str(repo))

import torch
import torchlens as tl
from torchvision.models import resnet18

assert Path(tl.__file__).resolve().is_relative_to(repo)
torch.manual_seed(0)

## Capture ResNet-18 and inspect the geometric table

In [ ]:
model = resnet18(weights=None).eval()
x = torch.randn(1, 3, 224, 224, requires_grad=True)
trace = tl.trace(model, x, backward_ready=True)

rf_table = trace.receptive_fields(level="layer").to_pandas()
rf_table[["name", "status", "size", "jump", "alignment"]].tail(16)

## The ResNet differentiator

The final spatial ResNet-18 feature has a 435 x 435 theoretical receptive-field hull. This is a graph result: the residual shortcuts are composed with the main paths rather than ignored by a sequential formula.

In [ ]:
target = next(op for op in trace.ops if op.layer_label == "layer4.1.conv2")
rf = target.receptive_field
assert rf.size == (435, 435)
rf.size, rf.jump, rf.center0, rf.status

## One unit: box, empirical support, and tripwire

`.at()` takes windowed-grid coordinates. `.gradient()` and `.check()` take a complete output-element index, including batch.

In [ ]:
box = rf.at((3, 3))
unit = rf.center_unit(batch_index=0)
gradient = rf.gradient(unit)
check = rf.check(unit)
assert check.passed
box, gradient.support_ranges, check.status

## Ancestor cone and input overlay

The graph callback highlights the actual receptive ancestor cone. The overlay combines the geometric box with the gradient heatmap.

In [ ]:
trace.draw(
    node_spec_fn=tl.receptive_field.node_spec(target, unit=unit),
    vis_fileformat="svg",
    vis_save_only=True,
)
overlay = rf.show(unit, gradient=True)
overlay

## Cross-validate the trace

The batch sweep and `verify()` use the same zero-tolerance gradient-inside-geometric tripwire.

In [ ]:
results = tl.receptive_field.cross_validate(trace, ops=[target], units="center")
verified = tl.receptive_field.verify(trace, ops=[target], units="center")
assert all(result.passed for result in results + verified)
[result.status for result in results]

## Projective and layer-to-layer geometry

In [ ]:
early = next(op for op in trace.ops if op.layer_label == "layer1.0.conv1")
projective = early.projective_field.at((8, 8), target=target)
same_projective = early.receptive_field.at((8, 8), direction="projective", target=target)
layer_to_layer = target.receptive_field.at((3, 3), source=early)
assert projective == same_projective
projective, layer_to_layer

In [ ]:
projective_table = trace.projective_fields(level="layer").to_pandas()
projective_table[["name", "status", "size", "jump"]].head()

# The integrated sampled validation runs both directions on a fresh capture.
validation_results = tl.validate(model, x, scope="receptive_field")
len(validation_results)

## ⚠️ GAPs / ergonomic smells

None observed in this audit. Keep any future failure visible here with the reproducing call and its error; do not replace a failed tripwire with a looser threshold.